# Agentic RAG với LangChain — Pháp Lý Việt Nam

Triển khai từ đầu một agent **Retrieval-Augmented Generation (RAG)** sử dụng giao diện tool-calling của LangChain. Agent thực hiện vòng lặp **ReAct** (Reasoning + Acting) để:
1. Tìm kiếm trong vector database nội bộ trước
2. Chuyển sang tìm kiếm web mô phỏng nếu kết quả nội bộ không đủ
3. Tổng hợp câu trả lời cuối chỉ khi được hỗ trợ bởi ngữ cảnh đã truy xuất

In [1]:
import os
import math
import json
from typing import List
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings


# --- Configuration Constants ---
LLM_URL = "https://api.openai.com/v1"
LLM_API_KEY = os.environ.get("LLM_API_KEY")
LLM_MODEL=os.environ.get("LLM_MODEL", "gpt-4o-mini")

## Bước 1 — Vector Store In-Memory

Một vector database tối giản (`TinyVectorDB`) xây dựng từ đầu. Sử dụng `HuggingFaceEmbeddings` (BAAI/bge-m3) để mã hóa tài liệu và độ tương đồng cosine để truy xuất — không cần thư viện vector DB bên ngoài.

In [2]:
# --- Step 1: Keep our From-Scratch Vector DB ---

def cosine_similarity(v1, v2):
    dot_product = sum(x * y for x, y in zip(v1, v2))
    magnitude1 = math.sqrt(sum(x * x for x in v1))
    magnitude2 = math.sqrt(sum(x * x for x in v2))
    return dot_product / (magnitude1 * magnitude2) if magnitude1 and magnitude2 else 0

class TinyVectorDB:
    def __init__(self):
        self.knowledge_base = []
        # Initialize LangChain embedding wrapper pointing to your endpoint
        self.encoder = HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            encode_kwargs={"normalize_embeddings": True}
        )

    def add_documents(self, docs: List[str]):
        embeddings = self.encoder.embed_documents(docs)
        for doc, emb in zip(docs, embeddings):
            self.knowledge_base.append({"text": doc, "embedding": emb})

    def query(self, query_text: str, top_k=2) -> List[str]:
        query_emb = self.encoder.embed_query(query_text)
        results = []
        for item in self.knowledge_base:
            sim = cosine_similarity(query_emb, item["embedding"])
            results.append((item["text"], sim))
        results.sort(key=lambda x: x[1], reverse=True)
        return [text for text, sim in results[:top_k]]

# Initialize the database instance
db = TinyVectorDB()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Bước 2 — Công Cụ Agent

Hai hàm được trang trí `@tool` của LangChain cung cấp khả năng truy xuất cho LLM:
- **`internal_vector_search`** — truy vấn vector DB in-memory chứa văn bản pháp luật Việt Nam
- **`web_search`** — mô phỏng tìm kiếm web dự phòng khi không tìm thấy trong cơ sở tri thức nội bộ

In [ ]:
# --- Bước 2: Định Nghĩa Công Cụ Agent LangChain ---

@tool
def internal_vector_search(query: str) -> str:
    """Tìm kiếm trong cơ sở tri thức nội bộ về văn bản pháp luật Việt Nam (luật, nghị định, thông tư)."""
    print(f"\n[Thực thi công cụ] Tìm kiếm vector DB cho: '{query}'")
    results = db.query(query, top_k=2)
    return json.dumps({"retrieved_docs": results}, ensure_ascii=False)

@tool
def web_search(query: str) -> str:
    """Công cụ dự phòng khi cơ sở tri thức nội bộ không có câu trả lời. Tìm kiếm thông tin pháp lý bên ngoài."""
    print(f"\n[Thực thi công cụ] Tìm kiếm web được kích hoạt cho: '{query}'")
    if "hiệu lực" in query.lower() or "còn hiệu lực" in query.lower():
        return "Theo Cổng thông tin pháp điển quốc gia, văn bản này hiện còn hiệu lực thi hành."
    return f"Kết quả tìm kiếm web cho '{query}': Không tìm thấy thông tin bên ngoài phù hợp."

tools = [internal_vector_search, web_search]
tools_map = {t.name: t for t in tools}

## Bước 3 — LLM với Công Cụ Liên Kết

Một instance `ChatOpenAI` được khởi tạo và `.bind_tools()` đăng ký các công cụ vào không gian hành động của model, cho phép nó phát ra các yêu cầu gọi công cụ có cấu trúc.

In [4]:
# --- Step 3: Define the Agent Engine ---

# Initialize your custom microservice LLM via LangChain
llm = ChatOpenAI(
    base_url=LLM_URL,
    model=LLM_MODEL,
    temperature=0
).bind_tools(tools) # Bind tools directly to the model's action capabilities

## Bước 4 — Vòng Lặp Agent ReAct

Vòng lặp suy luận cốt lõi chạy tối đa 5 lần lặp. Ở mỗi bước, LLM sẽ:
- **Gọi công cụ** → kết quả được thêm vào lịch sử tin nhắn và vòng lặp tiếp tục
- **Trả về câu trả lời cuối** → vòng lặp kết thúc và trả về kết quả

In [ ]:
# --- Bước 4: Vòng Lặp Logic ReAct của Agent ---

def run_agentic_rag(user_question: str):
    print(f"\n[Câu hỏi đầu vào]: '{user_question}'")

    messages = [
        SystemMessage(content=(
            "Bạn là hệ thống Agentic RAG chuyên về pháp luật Việt Nam. "
            "Bạn có quyền truy cập vào cơ sở tri thức nội bộ và web. "
            "Đầu tiên, hãy tìm kiếm vector database nội bộ bằng `internal_vector_search`. "
            "Đánh giá kết quả. Nếu không liên quan, hãy viết lại từ khóa tìm kiếm hoặc dùng `web_search`. "
            "Chỉ trả lời khi bạn có tài liệu ngữ cảnh hợp lệ hỗ trợ câu trả lời. "
            "Trả lời bằng tiếng Việt, ngắn gọn và trích dẫn nguồn điều khoản cụ thể."
        )),
        HumanMessage(content=user_question)
    ]

    for step in range(5):
        response = llm.invoke(messages)
        messages.append(response)

        if response.tool_calls:
            for tool_call in response.tool_calls:
                tool_name = tool_call["name"]
                tool_args = tool_call["args"]

                selected_tool = tools_map[tool_name]
                tool_output = selected_tool.invoke(tool_args)

                print(f"-> Agent nhận dữ liệu từ [{tool_name}]")

                messages.append({
                    "role": "tool",
                    "name": tool_name,
                    "tool_call_id": tool_call["id"],
                    "content": tool_output
                })
        else:
            print(f"\n[Kết quả cuối của Agent]: {response.content}")
            return response.content

    print("\n[Cảnh báo] Agent đã đạt số bước tối đa mà không kết luận được.")
    return None

## Bước 5 — Demo End-to-End

Hai truy vấn thử nghiệm kiểm tra toàn bộ pipeline:
1. **Truy xuất nội bộ** — câu hỏi về thời giờ làm việc được tìm thấy trong vector DB
2. **Dự phòng web** — câu hỏi về hiệu lực văn bản không có trong DB, chuyển sang công cụ tìm kiếm web

In [ ]:
# --- Bước 5: Kiểm Thử ---

db.add_documents([
    "Theo Điều 105 Bộ luật Lao động 2019, thời giờ làm việc bình thường không quá 8 giờ trong một ngày và không quá 48 giờ trong một tuần.",
    "Điều 107 Bộ luật Lao động 2019 quy định người lao động làm thêm giờ không được vượt quá 50% số giờ làm việc bình thường trong ngày; tổng số giờ làm việc và làm thêm không quá 12 giờ trong một ngày.",
    "Điều 111 Luật Doanh nghiệp 2020 quy định công ty cổ phần là doanh nghiệp có vốn điều lệ được chia thành cổ phần; số lượng cổ đông tối thiểu là 03 và không hạn chế số lượng tối đa.",
    "Khoản 1 Điều 166 Luật Đất đai 2013 quy định người sử dụng đất có quyền được cấp Giấy chứng nhận quyền sử dụng đất, quyền sở hữu nhà ở và tài sản khác gắn liền với đất.",
])

print("--- Bắt đầu Agentic RAG Pháp Lý Việt Nam với LangChain ---")

# Demo 1: Truy xuất nội bộ trực tiếp
run_agentic_rag("Người lao động được làm thêm tối đa bao nhiêu giờ mỗi ngày theo Bộ luật Lao động?")

print("\n" + "=" * 50)

# Demo 2: Dự phòng web
run_agentic_rag("Luật Đất đai 2013 hiện còn hiệu lực không?")